# EAIOS M8 · chat UI on the real Qwen gateway (browser test)

Choose **Runtime → Change runtime type → T4 GPU**, then run the cells in order.

Everything runs **inside this runtime over loopback**: the pinned model gateway (Qwen3-8B-AWQ, BGE-M3, development checker) on 127.0.0.1:8100, PostgreSQL/Redis/Qdrant, the backend API (`MODEL_MODE=gateway`) on 127.0.0.1:18000, the built web UI on 127.0.0.1:13000 and a headless Chromium (Playwright) that uses the UI like a person. No tunnel is opened and no port is exposed outside the runtime. Data is the synthetic NileTech fixture set.

The browser test passes only if:
1. a newly phrased annual-leave question (**Documents**) returns a cited answer and the citation opens the passage with *text hash verified*;
2. the sales reader gets the no-sources refusal for the restricted finance question, with no restricted figures on the page;
3. an open-ended **General** question gets a non-empty reply with no citations, labelled as general.

For 1 and 3 the run record must show `inference_mode = gateway`, the pinned `Qwen/Qwen3-8B-AWQ@4da05a8edb55c6046cce958586c33b61da07bb79`, and an actual model call (purpose `answer` / `general`, measured duration and token count).

Cell 5 downloads **`m8-chat-evidence-<UTC stamp>.zip`** (secret-scanned). Return that file.

In [ ]:
# @title 1 · Checkout the branch under review and verify the GPU
import os, subprocess
REPO = "/content/Enterprise-AI-OS"
BRANCH = "feature/m8-real-model-chat"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, "https://github.com/youssefhatembahig6-alt/Enterprise-AI-OS", REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "fetch", "-q", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO, "checkout", "-q", "FETCH_HEAD"], check=True)
subprocess.run(["git", "-C", REPO, "log", "-1", "--format=%H %s"], check=True)
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], check=True)

In [ ]:
# @title 2 · Install the pinned vLLM environment (about 5 minutes)
subprocess.run(["python", f"{REPO}/ml/colab-m8-chat/run_m8_chat_colab.py", "vllm-env"], check=True)

In [ ]:
# @title 3 · Launch the model gateway on loopback (waits until the model is served)
import secrets
os.environ.setdefault("GATEWAY_TOKEN", secrets.token_urlsafe(32))
%run /content/Enterprise-AI-OS/ml/model-gateway/colab_launch.py

In [ ]:
# @title 4 · Start the app, build the UI, run the browser test, bundle evidence (about 20–30 minutes)
subprocess.run(["python", f"{REPO}/ml/colab-m8-chat/run_m8_chat_colab.py", "all"], check=True)

In [ ]:
# @title 5 · Download the secret-scanned evidence bundle
import glob
from google.colab import files
bundles = sorted(glob.glob("/content/m8-chat-evidence-*.zip"))
if not bundles:
    raise RuntimeError("No evidence bundle exists; inspect the output of cell 4.")
files.download(bundles[-1])